<a href="https://colab.research.google.com/github/abhi-shek16/Youtube_chatboat/blob/main/youtube_chatboat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Pipeline using Groq + LangChain

This notebook builds a **Retrieval-Augmented Generation (RAG)** pipeline that:
1. Fetches a YouTube video transcript
2. Splits it into chunks
3. Embeds chunks using HuggingFace (free, local)
4. Stores embeddings in a FAISS vector store
5. Answers questions using **Groq's llama-3.3-70b** model

---
**Get your free Groq API key at:** https://console.groq.com/keys

## 🔑 Set your Groq API Key

In [2]:
import os
from google.colab import userdata

# Best practice: store your key in Colab Secrets (the 🔑 icon in the left sidebar)
# and access it like this:
try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ Groq API key loaded from Colab Secrets")
except Exception:
    # Fallback: paste your key directly (not recommended for shared notebooks)
    os.environ["GROQ_API_KEY"] = "your-groq-api-key-here"
    print("⚠️  Using hardcoded key — consider using Colab Secrets instead")

✅ Groq API key loaded from Colab Secrets


## 📦 Install Libraries

In [3]:
!pip install -q \
    youtube-transcript-api \
    langchain \
    langchain-community \
    langchain-groq \
    langchain-huggingface \
    langchain-text-splitters \
    faiss-cpu \
    sentence-transformers \
    tiktoken

print("✅ All libraries installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
✅ All libraries installed


## 📥 Imports

In [4]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

print("✅ Imports successful")

✅ Imports successful


## Step 1 — Fetch YouTube Transcript

Replace `video_id` with any YouTube video ID you want to query.

*(Default: Lex Fridman's interview with Demis Hassabis of DeepMind)*

In [5]:
video_id = "Gfr50f6ZBvo"  # Replace with your video ID

try:
    # Fix for newer youtube-transcript-api versions (>=0.7.0)
    ytt_api = YouTubeTranscriptApi()
    fetched = ytt_api.fetch(video_id)
    transcript = " ".join(snippet.text for snippet in fetched)
    print(f"✅ Transcript fetched — {len(transcript)} characters")
    print("\n--- Preview (first 500 chars) ---")
    print(transcript[:500])

except TranscriptsDisabled:
    print("❌ No captions available for this video.")
except Exception as e:
    print(f"❌ Error: {e}")

✅ Transcript fetched — 133836 characters

--- Preview (first 500 chars) ---
the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful 


## Step 2 — Split into Chunks

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.create_documents([transcript])

print(f"✅ Split into {len(chunks)} chunks")
print("\n--- Sample chunk ---")
print(chunks[0].page_content)

✅ Split into 168 chunks

--- Sample chunk ---
the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrot

## Step 3 — Create Embeddings & FAISS Vector Store

We use **HuggingFace's `all-MiniLM-L6-v2`** — a lightweight, high-quality embedding model that runs locally for free (no API key needed).

In [7]:
print("⏳ Loading embedding model (downloads ~90MB on first run)...")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("⏳ Building FAISS vector store...")
vector_store = FAISS.from_documents(chunks, embedding_model)

print("✅ Vector store ready!")

⏳ Loading embedding model (downloads ~90MB on first run)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

⏳ Building FAISS vector store...
✅ Vector store ready!


## Step 4 — Set Up Retriever

In [8]:
# Retrieve the top 4 most relevant chunks for each query
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("✅ Retriever ready")

# Quick test
test_docs = retriever.invoke("What is DeepMind working on?")
print(f"\n📄 Retrieved {len(test_docs)} chunks for test query")

✅ Retriever ready

📄 Retrieved 4 chunks for test query


## Step 5 — Set Up Groq LLM

Using **llama-3.3-70b-versatile** — Groq's highest quality model, runs at blazing speed.

In [9]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,           # 0 = deterministic, factual answers
    max_tokens=1024,
    api_key=os.environ["GROQ_API_KEY"]
)

print("✅ Groq LLM ready (llama-3.3-70b-versatile)")

✅ Groq LLM ready (llama-3.3-70b-versatile)


## Step 6 — Create Prompt Template

In [10]:
prompt_template = """
You are a helpful assistant. Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't have enough information to answer that."

Context:
{context}

Question: {question}

Answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

print("✅ Prompt template ready")

✅ Prompt template ready


## Step 7 — Build the RAG Chain

Wiring everything together with LangChain Expression Language (LCEL):

```
question → [retriever | format_docs]  ──┐
           [RunnablePassthrough]    ──┘ → prompt → llm → parser → answer
```

In [11]:
def format_docs(retrieved_docs):
    """Concatenate retrieved chunks into a single context string."""
    return "\n\n".join(doc.page_content for doc in retrieved_docs)


# Step 1: Retrieve context + pass question through in parallel
parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

# Step 2: Full chain — parallel → prompt → LLM → parse output
parser = StrOutputParser()
rag_chain = parallel_chain | prompt | llm | parser

print("✅ RAG chain assembled")

✅ RAG chain assembled


## 🚀 Ask Questions!

Run the cell below with any question about the video.

In [12]:
def ask(question):
    """Ask a question about the video and print the answer."""
    print(f"❓ Question: {question}")
    print("-" * 60)
    answer = rag_chain.invoke(question)
    print(f"🤖 Answer:\n{answer}")
    print("=" * 60)
    return answer


# --- Try some questions ---
ask("Who is Demis Hassabis?")

❓ Question: Who is Demis Hassabis?
------------------------------------------------------------
🤖 Answer:
Demis Hassabis is the CEO and co-founder of DeepMind, a company that has developed and published some of the most advanced artificial intelligence systems in the history of computing. He is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general.


'Demis Hassabis is the CEO and co-founder of DeepMind, a company that has developed and published some of the most advanced artificial intelligence systems in the history of computing. He is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general.'

In [13]:
ask("Is nuclear fusion discussed in the video?")

❓ Question: Is nuclear fusion discussed in the video?
------------------------------------------------------------
🤖 Answer:
Yes, nuclear fusion is discussed in the context.


'Yes, nuclear fusion is discussed in the context.'

In [14]:
ask("Can you summarize the main topics of this video?")

❓ Question: Can you summarize the main topics of this video?
------------------------------------------------------------
🤖 Answer:
The main topics of this conversation appear to be the search for a deeper and simpler explanation of physics, encompassing mysteries such as consciousness, life, and gravity, as well as the importance of clear and simple explanations as a sign of intelligence. Additionally, the conversation touches on the idea of enhancing human capabilities with technology and the relationship between humans and computers.


'The main topics of this conversation appear to be the search for a deeper and simpler explanation of physics, encompassing mysteries such as consciousness, life, and gravity, as well as the importance of clear and simple explanations as a sign of intelligence. Additionally, the conversation touches on the idea of enhancing human capabilities with technology and the relationship between humans and computers.'

In [15]:
# 💬 Try your own question!
your_question = "What does Demis think about consciousness?"  # ← change this
ask(your_question)

❓ Question: What does Demis think about consciousness?
------------------------------------------------------------
🤖 Answer:
Demis thinks that none of the current systems have any semblance of consciousness or sentience. He believes that discussing sentience in current systems is premature and that any appearance of sentience is a projection of human minds. He also mentions that consciousness itself has no agreed definition, but likes the working definition that "it's the way information feels when it gets processed". Additionally, he thinks that self-awareness is a necessary but not sufficient component of consciousness.


'Demis thinks that none of the current systems have any semblance of consciousness or sentience. He believes that discussing sentience in current systems is premature and that any appearance of sentience is a projection of human minds. He also mentions that consciousness itself has no agreed definition, but likes the working definition that "it\'s the way information feels when it gets processed". Additionally, he thinks that self-awareness is a necessary but not sufficient component of consciousness.'

---
## 🗺️ Pipeline Summary

| Step | Component | Tool Used |
|------|-----------|----------|
| Transcript Fetch | YouTube API | `youtube-transcript-api` |
| Chunking | Text Splitter | `RecursiveCharacterTextSplitter` |
| Embeddings | Local model | `HuggingFace all-MiniLM-L6-v2` |
| Vector Store | Similarity search | `FAISS` |
| LLM | Answer generation | `Groq llama-3.3-70b-versatile` |
| Chain | Orchestration | `LangChain LCEL` |